# Feature Engineering

## Objective
Convert validated enterprise tables into model-ready feature datasets with time-based features, encodings, scaling, and feature selection outputs.

The reusable production logic lives in `backend/app/features/feature_engineering.py`.

## Business Context

Feature engineering is where raw analytics become machine learning inputs. Time-aware, scale-aware, and category-aware transformations are essential for forecasting, churn prediction, risk scoring, and executive KPI prediction.

## Architecture and Implementation Plan

1. Load the processed, validated enterprise tables.
2. Build domain-specific feature datasets for customers, sales, finance, marketing, inventory, employees, and KPIs.
3. Apply encoding, scaling, lag, and rolling transformations.
4. Run feature selection utilities and persist the results.
5. Save feature datasets, schema, metadata, and selection reports for backend and model training use.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
backend_root = project_root / 'backend'
sys.path.insert(0, str(backend_root))

from app.features.feature_engineering import EnterpriseFeatureEngineer, FeatureEngineeringPaths

processed_dir = project_root / 'processed'
features_dir = project_root / 'features'
reports_dir = project_root / 'reports'
engineer = EnterpriseFeatureEngineer(FeatureEngineeringPaths(processed_dir=processed_dir, features_dir=features_dir, reports_dir=reports_dir))
engineer

## Build Feature Package

The next cell creates the model-ready feature datasets and writes the schema and metadata artifacts.

In [ ]:
feature_package = engineer.build_feature_package()
{name: frame.shape for name, frame in feature_package.items()}

## Feature Dataset Preview

Review the customer and sales feature sets because they are the primary inputs for forecasting and churn models.

In [ ]:
customer_features = feature_package['customer_features']
sales_features = feature_package['sales_features']
display(customer_features.head())
display(sales_features.head())

## Selection and Scaling Review

The bar chart below shows the number of features in each engineered dataset so we can confirm the transformed outputs are substantial and balanced.

In [ ]:
feature_counts = pd.DataFrame({
    'dataset': list(feature_package.keys()),
    'columns': [len(frame.columns) for frame in feature_package.values()],
})
ax = feature_counts.plot(kind='bar', x='dataset', y='columns', legend=False, figsize=(12, 4), color='#1f77b4')
ax.set_title('Engineered Feature Columns by Dataset')
ax.set_xlabel('Dataset')
ax.set_ylabel('Column Count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## Saved Artifacts

The feature engineering module saves all outputs to `features/` and `reports/`, including the feature schema, metadata, and selection report.

In [ ]:
artifact_preview = pd.DataFrame({
    'file': [
        'features/customer_features.csv',
        'features/sales_features.csv',
        'features/finance_features.csv',
        'features/marketing_features.csv',
        'features/inventory_features.csv',
        'features/employee_features.csv',
        'features/kpi_features.csv',
        'features/feature_schema.json',
        'features/feature_metadata.json',
        'reports/feature_selection_report.csv',
    ],
    'exists': [
        (features_dir / 'customer_features.csv').exists(),
        (features_dir / 'sales_features.csv').exists(),
        (features_dir / 'finance_features.csv').exists(),
        (features_dir / 'marketing_features.csv').exists(),
        (features_dir / 'inventory_features.csv').exists(),
        (features_dir / 'employee_features.csv').exists(),
        (features_dir / 'kpi_features.csv').exists(),
        (features_dir / 'feature_schema.json').exists(),
        (features_dir / 'feature_metadata.json').exists(),
        (reports_dir / 'feature_selection_report.csv').exists(),
    ]
})
artifact_preview

## Conclusions

The platform now has reusable, model-ready enterprise feature datasets and governance artifacts. These outputs can feed the forecasting, churn, risk, segmentation, and KPI modeling notebooks without rewriting transformation logic.